In [3]:
import pandas as pd

log = pd.read_csv('/Users/francescameneghello/Downloads/simulated_log_test_only_sepsis_0.1.csv')

In [4]:
log

In [6]:
log['resource'].unique().tolist()

In [7]:
def start_error(log):
    error = 0
    log['end_time'] = pd.to_datetime(log['end_time'])
    log['start_time'] = pd.to_datetime(log['start_time'])
    for index, row in log.iterrows():
        if row['start_time']>row['end_time']:
            error += 1
    return error

In [8]:
start_error(log)

In [11]:
### overlap

def overlap_error(log):
    log = log[log['resource'].notna()]
    res = log['resource'].unique().tolist()
    capacity = {r: 1 for r in res}
    overlap_error = 0
    resource_work = {r: {} for r in res}
    log['end_time'] = pd.to_datetime(log['end_time'])
    log['start_time'] = pd.to_datetime(log['start_time'])
    for index, row in log.iterrows():
        start = int(row['start_time'].timestamp())
        end = int(row['end_time'].timestamp())
        for i in range(start, end):
            if i in resource_work[row['resource']]:
                resource_work[row['resource']][i] += 1
                if resource_work[row['resource']][i] > capacity[row['resource']]:
                    overlap_error += 1
            else:
                resource_work[row['resource']][i] = 1
    return overlap_error

In [2]:
def find_overlapping_intervals(intervals):
    overlaps = []
    n = len(intervals)
    for i in range(n):
        for j in range(i + 1, n):
            a, b = intervals[i]
            c, d = intervals[j]
            # Check if intervals [a, b] and [c, d] overlap
            if max(a, c) <= min(b, d):
                overlaps.append([[a, b], [c, d]])
    return overlaps

# Example usage
intervals = [[2, 7], [5, 6], [9, 12], [3, 4]]
overlapping_pairs = find_overlapping_intervals(intervals)
print("Overlapping intervals:")
for pair in overlapping_pairs:
    print(pair)

Overlapping intervals:
[[2, 7], [5, 6]]
[[2, 7], [3, 4]]


In [3]:
log = pd.read_csv('/Users/francescameneghello/Documents/GitHub/nirdizati-light/predictive_models_processing_time/sepsis_estimated_start.csv', sep=";")
log

In [33]:
import pandas as pd

log = pd.read_csv('/Users/francescameneghello/Documents/GitHub/nirdizati-light/predictive_models_processing_time/BPI_Challenge_2012_estimated_start.csv', sep=",")
start_times = []
log['start:timestamp'] = log['start:timestamp'].astype(str).str[:19]
log['start:timestamp'] = pd.to_datetime(log['start:timestamp'], utc=True, format="%Y-%m-%d %H:%M:%S")
caseid_unique = list(log['caseid'].unique())
for caseid in caseid_unique:
    group_case = log[log['caseid'] == caseid].sort_values(by='start:timestamp')
    start_times.append(group_case.iloc[0]['start:timestamp'])

In [28]:
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from scipy import stats

# Example list of datetime objects (assumed to be defined elsewhere)
datetimes = sorted(start_times)

# Convert datetimes to numeric values (e.g., seconds since the first datetime)
time_deltas = np.array([
    (datetimes[i] - datetimes[i - 1]).total_seconds()
    for i in range(1, len(datetimes))
])

# --- Outlier removal using Z-score ---
z_scores = np.abs(stats.zscore(time_deltas))
threshold = 3
filtered_deltas = time_deltas[z_scores < threshold]

# List of distributions to test
distributions = ['norm', 'expon', 'lognorm', 'gamma', 'beta']
results = []

# Fit each distribution and compute the Kolmogorov-Smirnov (KS) statistic
x = np.linspace(min(filtered_deltas), max(filtered_deltas), 100)
hist_vals, bin_edges = np.histogram(filtered_deltas, bins=5, density=True)

for dist_name in distributions:
    dist = getattr(stats, dist_name)
    params = dist.fit(filtered_deltas)
    ks_stat, ks_pvalue = stats.kstest(filtered_deltas, dist_name, args=params)
    
    results.append({
        'distribution': dist_name,
        'params': params,
        'ks_stat': ks_stat,
        'ks_pvalue': ks_pvalue
    })

# Sort by KS statistic (lower is better)
results.sort(key=lambda r: r['ks_stat'])

# Plot the histogram and best 3 fitted PDFs
plt.hist(filtered_deltas, bins=5, density=True, alpha=0.5, label='Data histogram')

for result in results[:3]:
    dist = getattr(stats, result['distribution'])
    pdf = dist.pdf(x, *result['params'])
    plt.plot(x, pdf, label=f"{result['distribution']}")

plt.xlabel('Seconds since start')
plt.ylabel('Density')
plt.legend()
plt.title("Top 3 Fitted Distributions (Outliers Removed)")
plt.tight_layout()

results[:3]  # Show the best 3 fits


In [27]:
mu = 20203198.389895137/3600
sigma = 10528412.75221794/3600  # standard deviation
n_samples = 1000            # number of data points

# Generate samples
data = np.random.normal(loc=mu, scale=sigma, size=10)
data

In [29]:
36028/3600

In [3]:
import pm4py
log = pm4py.read_xes('/Users/francescameneghello/Documents/GitHub/nirdizati-light/datasets/BPI_Challenge_2017_W_Two_TS/full_label.xes')
log

/Users/francescameneghello/opt/anaconda3/lib/python3.9/site-packages/pm4py/util/dt_parsing/parser.py:82: UserWarning: ISO8601 strings are not fully supported with strpfromiso for Python versions below 3.11
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/30276 [00:00<?, ?it/s]

,concept:name,Resource,start:timestamp,time:timestamp,case:concept:name,case:RequestedAmount,case:LoanGoal,case:ApplicationType,case:label
0,W_Complete application,User_14,2016-08-04 13:39:29+00:00,2016-08-04 13:50:12+00:00,Application_1000086665,5000.0,"Other, see explanation",New credit,regular
1,W_Call after offers,User_5,2016-08-05 14:01:23+00:00,2016-08-05 14:03:01+00:00,Application_1000086665,5000.0,"Other, see explanation",New credit,regular
2,W_Call after offers,User_18,2016-08-09 18:25:01+00:00,2016-08-09 18:25:32+00:00,Application_1000086665,5000.0,"Other, see explanation",New credit,regular
3,W_Complete application,User_32,2016-06-06 08:02:16+00:00,2016-06-06 08:16:46+00:00,Application_1000158214,12500.0,Home improvement,New credit,deviant
4,W_Call after offers,User_32,2016-06-06 08:16:46+00:00,2016-06-06 08:18:42+00:00,Application_1000158214,12500.0,Home improvement,New credit,deviant
...,...,...,...,...,...,...,...,...,...
206377,W_Call after offers,User_42,2016-10-10 12:47:12+00:00,2016-10-10 12:47:52+00:00,Application_999993812,30000.0,Caravan / Camper,New credit,regular
206378,W_Call after offers,User_18,2016-10-14 08:18:35+00:00,2016-10-14 08:18:57+00:00,Application_999993812,30000.0,Caravan / Camper,New credit,regular
206379,W_Validate application,User_113,2016-10-19 10:28:03+00:00,2016-10-19 10:30:31+00:00,Application_999993812,30000.0,Caravan / Camper,New credit,regular
206380,W_Call incomplete files,User_109,2016-10-19 12:44:29+00:00,2016-10-19 12:44:35+00:00,Application_999993812,30000.0,Caravan / Camper,New credit,regular


In [13]:
start_times = []
caseid_unique = list(log['case:concept:name'].unique())
for caseid in caseid_unique:
    group_case = log[log['case:concept:name'] == caseid].sort_values(by='start:timestamp')
    start_times.append(group_case.iloc[0]['start:timestamp'])

KeyboardInterrupt: 

In [7]:
print({res: idx for idx, res in enumerate(list(log['Resource'].unique()))})

{'User_14': 0, 'User_5': 1, 'User_18': 2, 'User_32': 3, 'User_118': 4, 'User_71': 5, 'User_16': 6, 'User_132': 7, 'User_91': 8, 'User_119': 9, 'User_58': 10, 'User_121': 11, 'User_28': 12, 'User_3': 13, 'User_29': 14, 'User_75': 15, 'User_24': 16, 'User_25': 17, 'User_37': 18, 'User_116': 19, 'User_90': 20, 'User_100': 21, 'User_30': 22, 'User_15': 23, 'User_77': 24, 'User_53': 25, 'User_133': 26, 'User_67': 27, 'User_7': 28, 'User_23': 29, 'User_126': 30, 'User_68': 31, 'User_95': 32, 'User_99': 33, 'User_98': 34, 'User_81': 35, 'User_69': 36, 'User_72': 37, 'User_94': 38, 'User_51': 39, 'User_123': 40, 'User_78': 41, 'User_134': 42, 'User_42': 43, 'User_34': 44, 'User_48': 45, 'User_120': 46, 'User_102': 47, 'User_43': 48, 'User_13': 49, 'User_27': 50, 'User_4': 51, 'User_46': 52, 'User_87': 53, 'User_40': 54, 'User_49': 55, 'User_131': 56, 'User_62': 57, 'User_56': 58, 'User_41': 59, 'User_112': 60, 'User_10': 61, 'User_96': 62, 'User_63': 63, 'User_80': 64, 'User_31': 65, 'User_19'

In [9]:
print({str(idx): res for idx, res in enumerate(list(log['Resource'].unique()))})

{'0': 'User_14', '1': 'User_5', '2': 'User_18', '3': 'User_32', '4': 'User_118', '5': 'User_71', '6': 'User_16', '7': 'User_132', '8': 'User_91', '9': 'User_119', '10': 'User_58', '11': 'User_121', '12': 'User_28', '13': 'User_3', '14': 'User_29', '15': 'User_75', '16': 'User_24', '17': 'User_25', '18': 'User_37', '19': 'User_116', '20': 'User_90', '21': 'User_100', '22': 'User_30', '23': 'User_15', '24': 'User_77', '25': 'User_53', '26': 'User_133', '27': 'User_67', '28': 'User_7', '29': 'User_23', '30': 'User_126', '31': 'User_68', '32': 'User_95', '33': 'User_99', '34': 'User_98', '35': 'User_81', '36': 'User_69', '37': 'User_72', '38': 'User_94', '39': 'User_51', '40': 'User_123', '41': 'User_78', '42': 'User_134', '43': 'User_42', '44': 'User_34', '45': 'User_48', '46': 'User_120', '47': 'User_102', '48': 'User_43', '49': 'User_13', '50': 'User_27', '51': 'User_4', '52': 'User_46', '53': 'User_87', '54': 'User_40', '55': 'User_49', '56': 'User_131', '57': 'User_62', '58': 'User_56

In [11]:
print({str(idx): res for idx, res in enumerate(list(log['concept:name'].unique()))})

{'0': 'W_Complete application', '1': 'W_Call after offers', '2': 'W_Validate application', '3': 'W_Call incomplete files', '4': 'W_Handle leads', '5': 'W_Assess potential fraud', '6': 'W_Shortened completion', '7': 'W_Personal Loan collection'}


In [16]:
#"112.0": {"resources": ["112.0"]},
{str(res): {"resource": [str(res)]} for idx, res in enumerate(list(log['Resource'].unique()))}

{'User_14': {'resource': ['User_14']},
 'User_5': {'resource': ['User_5']},
 'User_18': {'resource': ['User_18']},
 'User_32': {'resource': ['User_32']},
 'User_118': {'resource': ['User_118']},
 'User_71': {'resource': ['User_71']},
 'User_16': {'resource': ['User_16']},
 'User_132': {'resource': ['User_132']},
 'User_91': {'resource': ['User_91']},
 'User_119': {'resource': ['User_119']},
 'User_58': {'resource': ['User_58']},
 'User_121': {'resource': ['User_121']},
 'User_28': {'resource': ['User_28']},
 'User_3': {'resource': ['User_3']},
 'User_29': {'resource': ['User_29']},
 'User_75': {'resource': ['User_75']},
 'User_24': {'resource': ['User_24']},
 'User_25': {'resource': ['User_25']},
 'User_37': {'resource': ['User_37']},
 'User_116': {'resource': ['User_116']},
 'User_90': {'resource': ['User_90']},
 'User_100': {'resource': ['User_100']},
 'User_30': {'resource': ['User_30']},
 'User_15': {'resource': ['User_15']},
 'User_77': {'resource': ['User_77']},
 'User_53': {'res

In [19]:
print({str(idx): res for idx, res in enumerate(list(log['case:ApplicationType'].unique()))})

{'0': 'New credit', '1': 'Limit raise'}


In [35]:
import pandas as pd
log = pd.read_csv('/Users/francescameneghello/Documents/GitHub/nirdizati-light/datasets/sepsis/sepsis_df_cf_0.2_pref_len_25.csv', sep=",")
log

,Diagnose,DiagnosticArtAstrup,DiagnosticBlood,DiagnosticECG,DiagnosticIC,DiagnosticLacticAcid,DiagnosticLiquor,DiagnosticOther,DiagnosticSputum,DiagnosticUrinaryCulture,...,prefix_25,CRP_25,LacticAcid_25,Leucocytes_25,Resource_25,duration_25,arrival_25,waiting_25,label,trace_id
0,SB,True,True,True,True,True,False,False,False,False,...,0,8.000000,0.5,1.500000,0,300.000000,0.0,9660.000000,deviant,VP
1,D,False,True,True,True,True,False,False,True,True,...,Leucocytes,251.999994,1.5,22.499999,B,3599.999965,0.0,86400.003222,deviant,CF
2,TB,False,True,True,True,True,False,False,False,True,...,0,8.000000,0.5,1.500000,0,300.000000,0.0,9660.000000,deviant,PG
3,E,True,True,True,True,True,False,False,False,True,...,0,8.000000,0.5,1.500000,0,300.000000,0.0,9660.000000,deviant,SI
4,TB,False,True,True,True,True,False,False,False,True,...,0,8.000000,0.5,1.500000,0,300.000000,0.0,9660.000000,deviant,AHA
5,SB,True,True,True,True,True,False,False,False,False,...,0,8.000000,0.5,1.500000,0,300.000000,0.0,9660.000000,deviant,BJA
6,CB,False,True,True,True,True,False,False,False,False,...,CRP,54.999998,1.9,6.400000,B,3599.999965,0.0,9660.000000,deviant,PIA
7,SB,True,True,True,True,True,False,False,False,False,...,0,8.000000,0.5,1.500000,0,300.000000,0.0,9660.000000,deviant,FX
8,A,other,other,other,other,other,other,other,other,other,...,0,8.000000,0.5,1.500000,0,300.000000,0.0,9660.000000,deviant,LW
9,SB,True,True,True,True,True,False,False,False,False,...,0,8.000000,0.5,1.500000,0,300.000000,0.0,9660.000000,deviant,CV


In [26]:
PARALLEL = ['LacticAcid', 'CRP', 'Leucocytes', 'IV Liquid', 'Admission IC', 'Admission NC', 'ER Sepsis Triage', 'IV Antibiotics']

def find_parallel(row):
    prefix_trace = []
    for index in range(1, 25):
        prefix = 'prefix_' + str(index)
        prefix_trace.append(1 if row[prefix] in PARALLEL else 0)
    print(prefix_trace)
    parallel_find = []
    index = 0
    open_sub = []
    while index<len(prefix_trace):
        if prefix_trace[index] == 1:
            if len(open_sub) == 0:
                open_sub = [index]
            else:
                open_sub.append(index)
        else:
            if len(open_sub) > 1:
                parallel_find.append(open_sub)
            open_sub = []
        index+=1
    if len(open_sub) > 0:
        parallel_find.append(open_sub)
    parallel_find = [[x + 1 for x in sublist] for sublist in parallel_find]
    return parallel_find

In [27]:
find_parallel(log.iloc[0])

[0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


[[3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]]

In [30]:
TRACE_ATTRIBUTES = ['InfectionSuspected',
       'DiagnosticBlood', 'DisfuncOrg', 'SIRSCritTachypnea', 'Hypotensie',
       'SIRSCritHeartRate', 'Infusion', 'DiagnosticArtAstrup',
       'DiagnosticIC', 'DiagnosticSputum', 'DiagnosticLiquor',
       'DiagnosticOther', 'SIRSCriteria2OrMore', 'DiagnosticXthorax',
       'SIRSCritTemperature', 'DiagnosticUrinaryCulture', 'SIRSCritLeucos',
       'Oligurie', 'DiagnosticLacticAcid', 'Diagnose', 'Hypoxie',
       'DiagnosticUrinarySediment', 'DiagnosticECG']
EVENT_ATTRIBUTES = ['Leucocytes', 'CRP', 'LacticAcid'] 

### event = [sequence/parallel, sequence/parallel, task, processing_time, resource, wait, 'label', attributes_event, attributes_trace
contrafactual = log
resource = 'Resource_'
columns = list(contrafactual.columns)
count_prefix = 1
contrafactual_traces = dict()
for index, row in contrafactual.iterrows():
    parallel_find = find_parallel(row)
    key = str(row['trace_id']) + "_CF"
    contrafactual_traces[key] = []
    prefix = 'prefix_' + str(count_prefix)
    attributes_trace = {}
    attributes_event = {}
    for k in TRACE_ATTRIBUTES:
        if k in row:
            attributes_trace[k] = row[k]
    for k in EVENT_ATTRIBUTES:
        attributes_event[k] = row[k + '_' + str(count_prefix)]
    post_last_parallel = 0
    while prefix in columns and row[prefix] != '0' and row[prefix] != 0:
        index = next((i for i, sub in enumerate(parallel_find) if count_prefix in sub), -1)
        if index > -1:
            head_of_parallel = parallel_find[index][0]
            if head_of_parallel == count_prefix:
                contrafactual_traces[key].append(
                    [True, row[prefix], -1, row[resource + str(count_prefix)], -1, row['label'], attributes_event, attributes_trace, []])
                post_last_parallel = len(contrafactual_traces[key])-1
            else:
                event = [False, row[prefix], -1, row[resource + str(count_prefix)], -1, row['label'], attributes_event, attributes_trace]
                contrafactual_traces[key][post_last_parallel][-1].append(event)
        else:
            contrafactual_traces[key].append(
                [False, row[prefix], -1, row[resource + str(count_prefix)], -1, row['label'], attributes_event, attributes_trace])
        count_prefix += 1
        prefix = 'prefix_' + str(count_prefix)
    count_prefix = 1

contrafactual_traces

[0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


{'VP_CF': [[False,
   'ER Registration',
   -1,
   'A',
   -1,
   'deviant',
   {'Leucocytes': 6.5, 'CRP': 11.0, 'LacticAcid': 0.0},
   {'InfectionSuspected': True,
    'DiagnosticBlood': True,
    'DisfuncOrg': False,
    'SIRSCritTachypnea': True,
    'Hypotensie': False,
    'SIRSCritHeartRate': True,
    'Infusion': True,
    'DiagnosticArtAstrup': True,
    'DiagnosticIC': True,
    'DiagnosticSputum': False,
    'DiagnosticLiquor': False,
    'DiagnosticOther': False,
    'SIRSCriteria2OrMore': True,
    'DiagnosticXthorax': True,
    'SIRSCritTemperature': True,
    'DiagnosticUrinaryCulture': False,
    'SIRSCritLeucos': False,
    'Oligurie': False,
    'DiagnosticLacticAcid': True,
    'Diagnose': 'SB',
    'Hypoxie': False,
    'DiagnosticUrinarySediment': False,
    'DiagnosticECG': True}],
  [False,
   'ER Triage',
   -1,
   'C',
   -1,
   'deviant',
   {'Leucocytes': 6.5, 'CRP': 11.0, 'LacticAcid': 0.0},
   {'InfectionSuspected': True,
    'DiagnosticBlood': True,
    'Di

In [36]:
#[[3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]]
prefix = 0
log = log[log["trace_id"] == "SI"]
for index in range(1, 25):
    prefix = 'prefix_' + str(index)
    print(index, log.iloc[0][prefix])

1 ER Registration
2 ER Triage
3 ER Sepsis Triage
4 IV Liquid
5 IV Antibiotics
6 LacticAcid
7 CRP
8 Leucocytes
9 Admission NC
10 Admission NC
11 Leucocytes
12 CRP
13 CRP
14 CRP
15 Leucocytes
16 CRP
17 Release A
18 Return ER
19 0
20 0
21 0
22 0
23 0
24 0


In [34]:
(3248.6*3)-6404.73

3341.0699999999997